In [1]:
"""
Deterministic unified salary determination analysis
(train.xlsx → model training; test.xlsx + test_salaries.xlsx → evaluation)

Implements exactly:
- Read all columns from default sheet (no filtering/reshaping of input files)
- Age from Born relative to fixed date 2024-07-01 (native date parsing)
- Test salaries merged by exact row order
- Position one-hot with alphabetically-first category as reference level
- Deterministic PCA (SVD) workload index using standardized TOI/GP, Shifts, Pace, OTOI
- Pearson correlation Salary vs workload index (train)
- Team-level Lerman–Yitzhaki weighted Gini (weights=GP), then unweighted mean across Teams
- Ridge regression (train only) with deterministic contiguous 5-fold CV, alpha by min mean MSE
- MAD of standardized residuals on merged test (standardize by train residual SD)
- Quantile regression at tau=0.90 via deterministic LP solver (HiGHS), coefficient on Age*TOI/GP
- Empirical Bayes hierarchical model (MixedLM) with Team random intercept, REML
  - report maximum random intercept posterior mean (and its Team)
  - variance ratio = Var(u) / (Var(u) + Var(fixed linear predictor))
- Split conformal 95% intervals using ridge, 80/20 row-order split of train
  - report mean interval width on merged test (constant width here)
- OLS slope of calibration fit: predicted Salary vs observed Salary on merged test
- Five visualizations saved as PNGs
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

import statsmodels.api as sm
from scipy.optimize import linprog

# ----------------------------
# File paths (edit as needed)
# ----------------------------
TRAIN_PATH = "train.xlsx"
TEST_PATH = "test.xlsx"
TEST_SALARIES_PATH = "test_salaries.xlsx"

# Fixed reference date for Age
REF_DATE = pd.Timestamp("2024-07-01")


# ----------------------------
# Helpers
# ----------------------------
def read_default_sheet_all_cols(path: str) -> pd.DataFrame:
    # default sheet, all columns
    return pd.read_excel(path)

def parse_born_to_age_years(born_series: pd.Series) -> pd.Series:
    """
    Parse Born using pandas' native parsing.
    Also handles Excel date serials if present.
    Age in years relative to REF_DATE using 365.25.
    """
    s = born_series

    # Try standard parsing (handles strings like '97-01-30' etc.)
    born = pd.to_datetime(s, errors="coerce")

    # If numbers (Excel serial days) and parsing failed, convert from Excel origin
    if born.isna().any() and np.issubdtype(s.dtype, np.number):
        born = pd.to_datetime(s, unit="D", origin="1899-12-30", errors="coerce")

    age = (REF_DATE - born).dt.days / 365.25
    return age

def to_float_df(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df[cols].apply(pd.to_numeric, errors="coerce").astype(float)
    out = out.replace([np.inf, -np.inf], np.nan)
    return out

def train_mean_impute(train_df: pd.DataFrame, other_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    means = train_df.mean(axis=0, skipna=True)
    return train_df.fillna(means), other_df.fillna(means)

def weighted_gini_lerman_yitzhaki(x: np.ndarray, w: np.ndarray) -> float:
    """
    Lerman–Yitzhaki weighted Gini:
      G = 2 * cov_w(x, r) / mean_w(x),
    where r is the fractional rank (weighted midpoint rank) in [0,1].
    """
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x = x[m]; w = w[m]
    if x.size == 0:
        return np.nan

    order = np.argsort(x)
    x = x[order]; w = w[order]

    W = w.sum()
    mu = np.sum(w * x) / W
    if mu == 0:
        return 0.0

    cumw = np.cumsum(w)
    p = cumw / W
    p_prev = np.concatenate(([0.0], p[:-1]))
    r = (p_prev + p) / 2.0  # midpoint rank

    cov = np.sum((w / W) * (x - mu) * (r - 0.5))
    return float(2.0 * cov / mu)

def contiguous_folds(n: int, k: int = 5) -> list[tuple[int,int]]:
    fold_sizes = [n // k + (1 if i < n % k else 0) for i in range(k)]
    starts = np.cumsum([0] + fold_sizes[:-1])
    return [(int(s), int(s + fs)) for s, fs in zip(starts, fold_sizes)]


# ----------------------------
# Load data (default sheet, all columns)
# ----------------------------
train = read_default_sheet_all_cols(TRAIN_PATH)
test = read_default_sheet_all_cols(TEST_PATH)
test_sal = read_default_sheet_all_cols(TEST_SALARIES_PATH)

# Merge test salaries by exact row order alignment
test_merged = test.copy()
if "Salary" in test_sal.columns:
    test_merged["Salary"] = test_sal["Salary"].values
else:
    test_merged["Salary"] = test_sal.iloc[:, 0].values  # fallback


# ----------------------------
# Age
# ----------------------------
train["Age"] = parse_born_to_age_years(train["Born"])
test_merged["Age"] = parse_born_to_age_years(test_merged["Born"])


# ----------------------------
# Position one-hot (alphabetically first as reference)
# ----------------------------
pos_cats = sorted(pd.Series(train["Position"].astype(str).unique()).tolist())
train["Position"] = pd.Categorical(train["Position"].astype(str), categories=pos_cats)
test_merged["Position"] = pd.Categorical(test_merged["Position"].astype(str), categories=pos_cats)

pos_dummies_train = pd.get_dummies(train["Position"], prefix="Position")
pos_dummies_test = pd.get_dummies(test_merged["Position"], prefix="Position")

ref_col = f"Position_{pos_cats[0]}"
for d in (pos_dummies_train, pos_dummies_test):
    if ref_col in d.columns:
        d.drop(columns=[ref_col], inplace=True)

# Align dummy columns deterministically
for c in pos_dummies_train.columns:
    if c not in pos_dummies_test.columns:
        pos_dummies_test[c] = 0
for c in pos_dummies_test.columns:
    if c not in pos_dummies_train.columns:
        pos_dummies_train[c] = 0
pos_dummies_train = pos_dummies_train.reindex(sorted(pos_dummies_train.columns), axis=1)
pos_dummies_test = pos_dummies_test.reindex(pos_dummies_train.columns, axis=1)


# ----------------------------
# (1) Workload index PC1 via deterministic SVD on standardized workload vars
# ----------------------------
work_vars = ["TOI/GP", "Shifts", "Pace", "OTOI"]

Xw_train_df = to_float_df(train, work_vars)
Xw_test_df = to_float_df(test_merged, work_vars)
Xw_train_df, Xw_test_df = train_mean_impute(Xw_train_df, Xw_test_df)

Xw_train = Xw_train_df.values
mu_w = Xw_train.mean(axis=0)
sd_w = Xw_train.std(axis=0, ddof=0)
sd_w = np.where(sd_w == 0, 1.0, sd_w)

Xw_train_std = (Xw_train - mu_w) / sd_w

# SVD (deterministic for given BLAS/LAPACK; note PC sign is inherently arbitrary)
U, S, Vt = np.linalg.svd(Xw_train_std, full_matrices=False)
loading1 = Vt[0, :]
work_index_train = Xw_train_std @ loading1

# Project to test with identical loadings/statistics
Xw_test = Xw_test_df.values
Xw_test_std = (Xw_test - mu_w) / sd_w
work_index_test = Xw_test_std @ loading1  # not required for corr, but computed per spec

salary_train = pd.to_numeric(train["Salary"], errors="coerce").astype(float).values
pearson_corr = float(np.corrcoef(salary_train, work_index_train)[0, 1])
pearson_corr_r4 = float(np.round(pearson_corr, 4))


# ----------------------------
# (2) Team-level weighted Gini (Lerman–Yitzhaki) with GP weights; average across teams
# ----------------------------
team_ginis = []
for team, df in train.groupby("Team", sort=True):
    x = pd.to_numeric(df["Salary"], errors="coerce").astype(float).values
    w = pd.to_numeric(df["GP"], errors="coerce").astype(float).values
    g = weighted_gini_lerman_yitzhaki(x, w)
    if np.isfinite(g):
        team_ginis.append(g)

avg_team_gini = float(np.mean(team_ginis))
avg_team_gini_r4 = float(np.round(avg_team_gini, 4))


# ----------------------------
# (3) Ridge regression with deterministic contiguous 5-fold CV
# ----------------------------
ridge_predictors_base = ["Age", "ixG", "iSF", "SCF", "TOI/GP", "OPS"]

X_train_base = to_float_df(train.assign(Age=train["Age"]), ridge_predictors_base)
X_test_base = to_float_df(test_merged.assign(Age=test_merged["Age"]), ridge_predictors_base)
X_train_base, X_test_base = train_mean_impute(X_train_base, X_test_base)

X_train = pd.concat([X_train_base.reset_index(drop=True), pos_dummies_train.reset_index(drop=True)], axis=1).astype(float)
X_test = pd.concat([X_test_base.reset_index(drop=True), pos_dummies_test.reset_index(drop=True)], axis=1).astype(float)

y_train = pd.to_numeric(train["Salary"], errors="coerce").astype(float).values
y_test = pd.to_numeric(test_merged["Salary"], errors="coerce").astype(float).values

n = len(y_train)
folds = contiguous_folds(n, k=5)

# Deterministic alpha grid (you can change the grid as long as it is deterministic)
alphas = np.logspace(-3, 6, 50)

def cv_mse(alpha: float) -> float:
    mses = []
    for a, b in folds:
        idx_val = np.arange(a, b)
        idx_tr = np.concatenate([np.arange(0, a), np.arange(b, n)])

        model = Pipeline([
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("ridge", Ridge(alpha=alpha, random_state=0)),
        ])
        model.fit(X_train.iloc[idx_tr], y_train[idx_tr])
        pred = model.predict(X_train.iloc[idx_val])
        mses.append(mean_squared_error(y_train[idx_val], pred))
    return float(np.mean(mses))

cv_mses = np.array([cv_mse(a) for a in alphas])
best_alpha = float(alphas[int(np.argmin(cv_mses))])

ridge_model = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("ridge", Ridge(alpha=best_alpha, random_state=0)),
])
ridge_model.fit(X_train, y_train)

pred_train = ridge_model.predict(X_train)
resid_train = y_train - pred_train
resid_sd_train = float(np.std(resid_train, ddof=1))  # residual SD (train)

# Residuals on merged test, standardized by train residual SD
pred_test = ridge_model.predict(X_test)
resid_test = y_test - pred_test
std_resid_test = resid_test / resid_sd_train

mad_std_resid = float(np.median(np.abs(std_resid_test - np.median(std_resid_test))))
mad_std_resid_r4 = float(np.round(mad_std_resid, 4))


# ----------------------------
# (4) Quantile regression tau=0.90 via deterministic LP solver
# ----------------------------
tau = 0.90

X_q = X_train.copy()
X_q["Age_x_TOIGP"] = X_train_base["Age"].values * X_train_base["TOI/GP"].values

Xq_mat = np.column_stack([np.ones(len(X_q)), X_q.values])  # intercept + predictors
yq = y_train

m, p = Xq_mat.shape
# Variables: beta (p) + u (m) + v (m)
c = np.concatenate([np.zeros(p), tau * np.ones(m), (1 - tau) * np.ones(m)])

# Constraints: Xb + u - v = y
A_eq = np.hstack([Xq_mat, np.eye(m), -np.eye(m)])
b_eq = yq
bounds = [(None, None)] * p + [(0, None)] * (2 * m)

lp_res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
beta_hat = lp_res.x[:p]

col_names = ["Intercept"] + list(X_q.columns)
interaction_coef = float(beta_hat[col_names.index("Age_x_TOIGP")])
interaction_coef_r4 = float(np.round(interaction_coef, 4))


# ----------------------------
# (5) Hierarchical model: Team random intercept (REML), fixed effects same as ridge predictors
# ----------------------------
X_fe = sm.add_constant(X_train, has_constant="add")
exog = X_fe.values.astype(float)
endog = y_train.astype(float)
groups = train["Team"].astype(str).values

mixed = sm.MixedLM(endog=endog, exog=exog, groups=groups)
mixed_res = mixed.fit(reml=True, method="lbfgs", disp=False)

# Random intercept variance (handle boundary/singularity)
try:
    var_u = float(mixed_res.cov_re.iloc[0, 0])
    if not np.isfinite(var_u) or var_u < 1e-12:
        var_u = 0.0
except Exception:
    var_u = 0.0

beta_fe = np.asarray(mixed_res.fe_params, dtype=float)
fixed_part = exog @ beta_fe
var_fixed = float(np.var(fixed_part, ddof=0))
variance_ratio = float(var_u / (var_u + var_fixed)) if (var_u + var_fixed) > 0 else np.nan
variance_ratio_r4 = float(np.round(variance_ratio, 4))

# Posterior means of random intercepts (may fail if covariance singular)
max_team_name = None
max_team_premium = 0.00
try:
    re = mixed_res.random_effects  # dict: team -> array([intercept])
    max_team_name, max_ri = None, -np.inf
    for team, val in re.items():
        ri = float(np.asarray(val)[0])
        if ri > max_ri:
            max_ri = ri
            max_team_name = team
    max_team_premium = float(np.round(max_ri, 2))
except Exception:
    # If singular (cannot compute BLUPs), premiums collapse to 0
    max_team_name = sorted(pd.Series(groups).unique().tolist())[0]
    max_team_premium = 0.00


# ----------------------------
# (6) Split conformal intervals (95%) using ridge; 80/20 row-order split of train
# ----------------------------
n_tr = int(np.floor(0.80 * n))
idx_tr = np.arange(0, n_tr)
idx_cal = np.arange(n_tr, n)

X_tr_sc = X_train.iloc[idx_tr]
y_tr_sc = y_train[idx_tr]
X_cal_sc = X_train.iloc[idx_cal]
y_cal_sc = y_train[idx_cal]

# Re-select alpha on training subset with contiguous folds (deterministic)
folds_sc = contiguous_folds(len(y_tr_sc), k=5)

def cv_mse_sc(alpha: float) -> float:
    mses = []
    n_sc = len(y_tr_sc)
    for a, b in folds_sc:
        idx_val = np.arange(a, b)
        idx_tr2 = np.concatenate([np.arange(0, a), np.arange(b, n_sc)])
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha, random_state=0)),
        ])
        model.fit(X_tr_sc.iloc[idx_tr2], y_tr_sc[idx_tr2])
        pred = model.predict(X_tr_sc.iloc[idx_val])
        mses.append(mean_squared_error(y_tr_sc[idx_val], pred))
    return float(np.mean(mses))

cv_mses_sc = np.array([cv_mse_sc(a) for a in alphas])
best_alpha_sc = float(alphas[int(np.argmin(cv_mses_sc))])

ridge_sc = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=best_alpha_sc, random_state=0)),
])
ridge_sc.fit(X_tr_sc, y_tr_sc)

pred_cal = ridge_sc.predict(X_cal_sc)
scores = np.abs(y_cal_sc - pred_cal)

# 95% quantile with deterministic "higher" rule
qhat = float(np.quantile(scores, 0.95, method="higher"))

# Interval width per point is 2*qhat (constant across test)
mean_width = float(2.0 * qhat)
mean_width_r2 = float(np.round(mean_width, 2))


# ----------------------------
# (7) OLS slope: predicted Salary vs observed Salary on merged test
# ----------------------------
# Fit: predicted = a + b * observed
ols_fit = sm.OLS(pred_test, sm.add_constant(y_test)).fit()
cal_slope = float(ols_fit.params[1])
cal_slope_r4 = float(np.round(cal_slope, 4))


# ----------------------------
# Print required scalar outputs (atomic scalar values)
# ----------------------------
print("Pearson corr(Salary, workload PC1) [train], r4:", pearson_corr_r4)
print("Avg Team weighted Gini (Lerman–Yitzhaki, weights=GP) [train], r4:", avg_team_gini_r4)
print("Ridge best alpha (contiguous 5-fold CV) [train], scalar:", best_alpha)
print("MAD of standardized ridge residuals [merged test], r4:", mad_std_resid_r4)
print("Quantile reg tau=0.90 coef(Age*TOI/GP) [train], r4:", interaction_coef_r4)
print("Max Team random intercept posterior mean [train], r2:", max_team_premium, "Team:", max_team_name)
print("Mean width of 95% split conformal intervals [merged test], r2:", mean_width_r2)
print("OLS slope: predicted vs observed [merged test], r4:", cal_slope_r4)
print("Variance ratio (RE var / (RE var + fixed var)) [train], r4:", variance_ratio_r4)


# ----------------------------
# Visualizations (saved as PNG)
# ----------------------------
# (1) Boxplot of Salary by Position on train
plt.figure()
data_pos = [pd.to_numeric(train.loc[train["Position"] == p, "Salary"], errors="coerce").astype(float).values for p in pos_cats]
plt.boxplot(data_pos, labels=pos_cats, showfliers=False)
plt.ylabel("Salary")
plt.xlabel("Position")
plt.title("Salary by Position (Train)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("viz1_boxplot_salary_by_position.png", dpi=200)
plt.close()

# (2) Pearson correlation heatmap of Salary with TOI/GP, ixG, iSF, SCF on train
plt.figure()
vars_corr = ["Salary", "TOI/GP", "ixG", "iSF", "SCF"]
corr_mat = train[vars_corr].apply(pd.to_numeric, errors="coerce").astype(float).corr().values
plt.imshow(corr_mat, aspect="equal")
plt.colorbar()
plt.xticks(range(len(vars_corr)), vars_corr, rotation=45, ha="right")
plt.yticks(range(len(vars_corr)), vars_corr)
plt.title("Pearson Correlation Heatmap (Train)")
for i in range(len(vars_corr)):
    for j in range(len(vars_corr)):
        plt.text(j, i, f"{corr_mat[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("viz2_corr_heatmap.png", dpi=200)
plt.close()

# (3) Scatter: workload index vs Salary on train
plt.figure()
plt.scatter(work_index_train, salary_train)
plt.xlabel("Workload Index (PC1)")
plt.ylabel("Salary")
plt.title("Workload Index vs Salary (Train)")
plt.tight_layout()
plt.savefig("viz3_workload_vs_salary.png", dpi=200)
plt.close()

# (4) Calibration plot: observed vs predicted Salary on merged test
plt.figure()
plt.scatter(pred_test, y_test)
mn = float(min(pred_test.min(), y_test.min()))
mx = float(max(pred_test.max(), y_test.max()))
plt.plot([mn, mx], [mn, mx])  # 45-degree line

# Also plot OLS fit: observed ~ predicted (optional visual)
ols_obs_on_pred = sm.OLS(y_test, sm.add_constant(pred_test)).fit()
a_line, b_line = ols_obs_on_pred.params
grid = np.linspace(mn, mx, 100)
plt.plot(grid, a_line + b_line * grid)

plt.xlabel("Predicted Salary")
plt.ylabel("Observed Salary")
plt.title("Calibration Plot (Test)")
plt.tight_layout()
plt.savefig("viz4_calibration_plot.png", dpi=200)
plt.close()

# (5) Violin plot of Salary by Team on train
plt.figure()
teams_sorted = sorted(train["Team"].astype(str).unique().tolist())
data_team = [pd.to_numeric(train.loc[train["Team"].astype(str) == t, "Salary"], errors="coerce").astype(float).values for t in teams_sorted]
plt.violinplot(data_team, showmeans=False, showmedians=True, showextrema=False)
plt.xticks(range(1, len(teams_sorted) + 1), teams_sorted, rotation=90)
plt.ylabel("Salary")
plt.xlabel("Team")
plt.title("Salary by Team (Train)")
plt.tight_layout()
plt.savefig("viz5_violin_salary_by_team.png", dpi=200)
plt.close()

print("\nSaved plots:")
print("viz1_boxplot_salary_by_position.png")
print("viz2_corr_heatmap.png")
print("viz3_workload_vs_salary.png")
print("viz4_calibration_plot.png")
print("viz5_violin_salary_by_team.png")


/tmp/ipython-input-3601641830.py:64: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  born = pd.to_datetime(s, errors="coerce")
/tmp/ipython-input-3601641830.py:64: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  born = pd.to_datetime(s, errors="coerce")
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE ma

Pearson corr(Salary, workload PC1) [train], r4: 0.5987
Avg Team weighted Gini (Lerman–Yitzhaki, weights=GP) [train], r4: 0.1868
Ridge best alpha (contiguous 5-fold CV) [train], scalar: 138.9495494373139
MAD of standardized ridge residuals [merged test], r4: 0.5534
Quantile reg tau=0.90 coef(Age*TOI/GP) [train], r4: 18270.5449
Max Team random intercept posterior mean [train], r2: 0.0 Team: ANA
Mean width of 95% split conformal intervals [merged test], r2: 8242737.19
OLS slope: predicted vs observed [merged test], r4: 0.461
Variance ratio (RE var / (RE var + fixed var)) [train], r4: 0.0


/tmp/ipython-input-3601641830.py:419: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_pos, labels=pos_cats, showfliers=False)



Saved plots:
viz1_boxplot_salary_by_position.png
viz2_corr_heatmap.png
viz3_workload_vs_salary.png
viz4_calibration_plot.png
viz5_violin_salary_by_team.png
